In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully!")

# diagnostik
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

# tampilan
sns.set_style("whitegrid")
%matplotlib inline

print("Libraries loaded successfully!")


Libraries loaded successfully!
Libraries loaded successfully!


In [2]:
# =========================
# Imports
# =========================
import numpy as np
import pandas as pd
import statsmodels.api as sm

# =========================
# Data generation
# =========================
np.random.seed(42)

n = 5000  # number of households

df = pd.DataFrame({
    "household_id": range(1, n + 1),
    "village": np.random.choice(["Village A", "Village B", "Village C"], n),
    "monthly_income": np.random.normal(1_500_000, 500_000, n).clip(300_000),
    "house_condition": np.random.choice(
        ["poor", "average", "good"], n, p=[0.45, 0.35, 0.20]
    ),
    "num_dependents": np.random.randint(0, 6, n)
})

print("Total rows:", len(df))

df.to_csv("dummy_social_assistance_targeting.csv", index=False)

# =========================
# Eligibility (ground truth)
# =========================
df["eligible_actual"] = (
    (df["monthly_income"] < 1_200_000) &
    (df["house_condition"] == "poor") &
    (df["num_dependents"] >= 2)
).astype(int)

# =========================
# Logistic Regression
# =========================
X = df[["monthly_income", "num_dependents"]]
X = sm.add_constant(X)

y = df["eligible_actual"]

logit_model = sm.Logit(y, X).fit(disp=False)

print(logit_model.summary())

# =========================
# Simulate targeting errors
# =========================
df["received_assistance"] = np.where(
    df["eligible_actual"] == 1,
    np.random.choice([1, 0], n, p=[0.7, 0.3]),   # exclusion error
    np.random.choice([0, 1], n, p=[0.85, 0.15])  # inclusion error
)

df["targeting_status"] = np.select(
    [
        (df["eligible_actual"] == 1) & (df["received_assistance"] == 1),
        (df["eligible_actual"] == 1) & (df["received_assistance"] == 0),
        (df["eligible_actual"] == 0) & (df["received_assistance"] == 1)
    ],
    ["accurate", "exclusion_error", "inclusion_error"],
    default="accurate"
)

print(df.head())
print("\nTargeting summary:")
print(df["targeting_status"].value_counts())

# Export the updated 5000-row dataframe to your CSV file

Total rows: 5000
                           Logit Regression Results                           
Dep. Variable:        eligible_actual   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4997
Method:                           MLE   Df Model:                            2
Date:                Sat, 14 Feb 2026   Pseudo R-squ.:                  0.3451
Time:                        22:32:04   Log-Likelihood:                -875.56
converged:                       True   LL-Null:                       -1336.9
Covariance Type:            nonrobust   LLR p-value:                4.214e-201
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.5854      0.205      2.861      0.004       0.184       0.986
monthly_income -3.929e-06   1.81e-07    -21.730      0.000   -4.28e-06   -3.57e-06
num_dependents     

In [3]:
# 1 = mis-targeted, 0 = correctly targeted
df["mis_targeted"] = df["targeting_status"].apply(
    lambda x: 1 if x in ["inclusion_error", "exclusion_error"] else 0
)

df["mis_targeted"].value_counts()


mis_targeted
0    4211
1     789
Name: count, dtype: int64

In [4]:
df.groupby("mis_targeted")[["monthly_income", "num_dependents"]].mean()
pd.crosstab(df["house_condition"], df["mis_targeted"], normalize="index")



mis_targeted,0,1
house_condition,,
average,0.866933,0.133067
good,0.852674,0.147326
poor,0.818423,0.181577


In [5]:
import statsmodels.api as sm

X = df[["monthly_income", "num_dependents"]]
X = sm.add_constant(X)

y = df["mis_targeted"]

logit_model = sm.Logit(y, X).fit()
logit_model.summary()


Optimization terminated successfully.
         Current function value: 0.435122
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           mis_targeted   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4997
Method:                           MLE   Df Model:                            2
Date:                Sat, 14 Feb 2026   Pseudo R-squ.:                0.002022
Time:                        22:32:04   Log-Likelihood:                -2175.6
converged:                       True   LL-Null:                       -2180.0
Covariance Type:            nonrobust   LLR p-value:                   0.01219
==================================================================================
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -1.3987      0.133    -10.490      0.000      -1.660      -1.137
monthly_income -2.224e-07   7.88e-08     -2.822      0.005   -3.77e-07   -6.79e-08
num_dependents     0.0219      0.023      0.970      0.332      -0.022       0.066
==================================================================================
"""

## Logistic Regression Interpretation

The dependent variable is **mis_targeted**, indicating whether a household
experienced mistargeting in social assistance distribution.

Key findings:
- **Monthly income** has a negative coefficient, suggesting higher income
  households tend to have lower probability of mistargeting, although the
  effect is not statistically significant.
- **Number of dependents** has a positive coefficient, indicating households
  with more dependents are more likely to experience mistargeting.
- The model converges successfully, indicating stable estimation.
